# Decoding as Search: Greedy and Beam Search

A language model gives every string a probability; *decoding* picks one. The most common choice is the **mode**, the
single most probable string. It is also the hardest to compute, so in practice we run greedy decoding or beam search
and hope they find it. This notebook asks two questions:
- when do greedy and beam search fail to find the mode (**search error**)?
- when is the mode itself the wrong answer (**model error**)?

The picture throughout is a tree. Each prefix is a node, each next token an edge with cost $-\log p$, and each finished
string a path from the root to a $\texttt{</s>}$ leaf. The mode is the cheapest such path. Exact search is a
shortest-path algorithm, and greedy and beam search are the same algorithm with most of its frontier thrown away.

We measure all of this on two engines trained on the finance corpus:
- **Topic 3's Kneser–Ney trigram.** Its prefixes recombine on their last two words, so Dijkstra finds its exact mode.
- **Topic 13's toy causal transformer.** Nothing recombines, so exact search is a best-first tree search.

The prompts are the 188 distinct three-word openings of held-out MD&A sentences.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "beam_search_decoding.py").exists() else here / "notebooks" / "beam-search-decoding"))
import beam_search_decoding as bs

S = bs.summary()
print(S["n_prompts"], "prompts; held-out continuation length", round(S["reference_length"], 3), "words")
for name in ("trigram", "decoder"):
    print(f"{name:8} exact mode: {S[f'{name}_mode_length']:.3f} words, empty on {S[f'{name}_mode_empty']:.3f};",
          f"greedy search error {S[f'{name}_a0_k1_search_error']:.3f}, beam(64) {S[f'{name}_a0_k64_search_error']:.3f}")

## 1. Decoding as a shortest path

**Definition 1 (the decoding problem).** A language model over a vocabulary $V$ with end mark $\texttt{</s>}$ assigns a
prompt $x$ and a string $y = (y_1, \dots, y_n)$ with $y_n = \texttt{</s>}$ the probability
$$p(y \mid x) = \prod_{t=1}^{n} p(y_t \mid x, y_{\lt t}).$$
The **mode** is $y^\star = \arg\max_y p(y \mid x)$. The model is *consistent* if these probabilities sum to 1. Then
only finitely many strings have probability above any $\varepsilon > 0$, so the mode exists.

**Proposition 1.** Give the edge from prefix $h$ to $hw$ the weight $w(h, w) = -\log p(w \mid x, h) \ge 0$. Then
$-\log p(y \mid x)$ is the total weight of $y$'s root-to-leaf path. So the mode is the cheapest path from the root to a
$\texttt{</s>}$ leaf.

*Proof.* Take $-\log$ of the product in Definition 1. Each factor is at most 1, so each weight is non-negative. $\square$

Since weights are non-negative, a prefix's cost $c(h) = -\log \pi(h)$ never falls as the prefix grows. Here
$\pi(h)$ is the probability that the string begins with $h$.

**Theorem 1 (uniform-cost search is exact).** Uniform-cost search keeps a frontier of prefixes, starting from the
root. It repeatedly removes the cheapest prefix: if that prefix ends in $\texttt{</s>}$ it is returned, otherwise its
children join the frontier. For a consistent model, the string returned is a mode. If $\mathbb E|Y| \lt \infty$,
the search pops at most $(2 + \mathbb E|Y|)/p(y^\star)$ prefixes.

*Proof.*
- **Invariant.** Every finished string not yet popped has a prefix on the frontier. This holds at the start, since the
  root is a prefix of every string. When $h$ is popped and its children are pushed, every string through $h$ gets a
  prefix among the children.
- **Exactness.** Suppose the first finished string popped is $h^\star$, at cost $c$. Every frontier entry costs at
  least $c$. Each unpopped finished string $y$ has a frontier prefix of cost at most $c(y)$, so $c(y) \ge c$.
- **Work.** Pops come out in non-decreasing cost, so every popped prefix has $\pi(h) \ge p(y^\star)$. The prefixes of
  length $t$ have total probability $\Pr(|Y| \ge t)$. So at most $\Pr(|Y| \ge t)/p(y^\star)$ of them are that probable.
  Summing over $t \ge 0$ gives $(1 + \mathbb E|Y|)/p(y^\star)$ open prefixes. At most $1/p(y^\star)$ finished strings
  are that probable, which gives the bound. $\square$

The implementation pushes children lazily. Popping a node pushes only its best child and its next sibling, so the heap
stays small even over a vocabulary of 3,427 words. This preserves the invariant: an unpushed sibling is never cheaper
than the sibling standing in for it.

**Corollary (recombination).** In an order-$n$ Markov model, $p(\cdot \mid h)$ depends only on the state $\sigma(h)$,
the last $n - 1$ tokens. If $\sigma(h) = \sigma(h')$ and $c(h) \le c(h')$, then every completion $z$ satisfies
$c(hz) - c(h'z) = c(h) - c(h') \le 0$, so $h'$ can be discarded. Uniform-cost search then becomes Dijkstra's algorithm on
the state graph, which has at most $(|V| + 1)^{n-1}$ nodes. Exact decoding is polynomial in the size of that graph; the
length-layered version is Viterbi's algorithm.

For a neural network nothing recombines. Chen et al. (2018) show that for a general RNN, deciding whether some string
has probability above $c$ is undecidable (their Thm 8). For a consistent RNN it is decidable, but the best string can be
longer than any computable bound (Thms 9–10). Restricted to polynomial length it is NP-complete (Thm 11).

In [ ]:
bs.test_uniform_cost_is_exact()
bs.test_trigram_graph_search_equals_tree_search()
for name in ("trigram", "decoder"):
    m = bs.mode_stats(name)
    print(f"{name:8} pops to the exact mode: median {m['pops_median']:.0f}, max {m['pops_max']}")

## 2. Greedy and beam search: pruned uniform-cost search

**Definition 2.** At each step, **beam search** with width $k$ forms every child of every open prefix and keeps the
$k$ most probable children. Kept children that end in $\texttt{</s>}$ move to a finished set, and the rest stay open.
Search stops when the best finished string scores at least as high as the best open prefix: no extension can do better
than its prefix. **Greedy decoding** is beam(1).

**Theorem 2 (no approximation guarantee).**
- **(a)** For every $\varepsilon > 0$ there is a model on which greedy's string has less than $\varepsilon$ times the
  mode's probability.
- **(b)** For every $k$ there is a model on which beam($k$) misses the mode and beam($k + 1$) finds it.
- **(c)** Beam($k + 1$) can return a strictly less probable string than beam($k$).

*Proof.* Each part is a construction.
- **(a)** The first token is $a$ (0.6) or $b$ (0.4). After $a$ come $m$ equally likely words, each followed by
  $\texttt{</s>}$. After $b$ comes $\texttt{</s>}$. Greedy takes $a$ and returns probability $0.6/m$. The mode is
  $b\,\texttt{</s>}$ at 0.4. The ratio $1.5/m$ is below $\varepsilon$ once $m \gt 1.5/\varepsilon$.
- **(b)** Take $k$ decoys, each with probability $(1 - p_a)/k$, and a word $a$ with $p_a = 0.9/(k+1)$. Each decoy is a
  little likelier than $a$. Each decoy forks into two words, and every string then ends.
  - Beam($k$) keeps the decoys. All of their strings have probability $(1 - p_a)/(2k) \lt p_a$.
  - Beam($k+1$) also keeps $a$. Its next candidate, $a\,\texttt{</s>}$, outranks everything, and the search stops on
    it.
- **(c)** After $a$ (0.6) come ten words of 0.1, each ending the string. After $b$ (0.4) come $c$ and $d$ (0.5 each),
  then 50 words of 0.02.
  - Greedy returns 0.06.
  - At step 2, beam(2)'s candidates $bc, bd$ (0.2) push out every $ax_i$ (0.06). It returns 0.004.
  - Beam(3) keeps one $ax_i$ and returns 0.06. $\square$

In [ ]:
bs.test_greedy_has_no_approximation_guarantee()
bs.test_beam_k_misses_what_beam_k_plus_one_finds()
bs.test_beam_is_not_monotone_in_k()
for name in ("trigram", "decoder"):
    print(f"{name:8} prompts where a wider beam returned a worse string: {bs.nonmonotone_share(name):.3f}")

## 3. A certificate

Huang, Zhao, and Ma (2017) stop beam search once the best finished string beats every open prefix. They call the
result optimal "modulo beam size": optimal among the strings the beam kept. One more number makes the guarantee
global: the best candidate the beam ever **pruned**.

**Theorem 3 (certificate).** Let $f$ be beam search's best finished string. Let $m$ be the best score of any
candidate it pruned, and $o$ the best score of a prefix open when it stopped. If $\log p(f) \ge \max(m, o)$, then $f$
is a mode.

*Proof.* Consider any finished string $y$. At each step its prefix was either kept or pruned, and if the prefix was
kept it is now finished or still open. So exactly one of three things holds:
- $y$ finished inside the beam, so it scores at most $f$;
- $y$ extends a pruned candidate, so it scores at most $m \le \log p(f)$;
- $y$ extends a prefix still open, so it scores at most $o \le \log p(f)$.

In the last two cases, scores only fall along a path. $\square$

The certificate is sufficient, not necessary. It can stay silent on a correct answer, when the beam pruned a prefix more
probable than the answer whose strings all turned out worse.

**Length normalization** (Wu et al. 2016) ranks finished strings by $\log p(y)/\mathrm{lp}(|y|)$, with
$\mathrm{lp}(n) = ((5 + n)/6)^{\alpha}$. Then a longer string can outscore its prefix, and both Theorem 1 and
Theorem 3 lose their footing. With a length cap $T$ the certificate survives. A completion $y$ of $h$ has
$\log p(y) \le \log p(h) \le 0$ and $\mathrm{lp}(|y|) \le \mathrm{lp}(T)$, so its score is at most
$\log p(h)/\mathrm{lp}(T)$. Comparing against $m/\mathrm{lp}(T)$ and $o/\mathrm{lp}(T)$ is again a proof.

In [ ]:
bs.test_certificate_is_sound()
bs.test_lab_tree()
for name in ("trigram", "decoder"):
    rows = bs.curve(name, 0.0)["rows"]
    print(f"{name:8} certified share by k:", {r["k"]: round(r["certified"], 3) for r in rows})

## 4. Length bias

**Theorem 4.**
- **(a)** If $\texttt{</s>}$ is the likeliest next token after $h$, then no completion of $h$ beats stopping:
  $\max_z p(hz) = p(h\,\texttt{</s>})$.
- **(b)** If every step's probability is at most $q \lt 1$, then $p(y) \le q^{|y|}$, where $|y|$ counts the words
  before $\texttt{</s>}$. So $|y^\star| \le \log p(y')/\log q$ for any finished $y'$.

*Proof.*
- **(a)** Take a completion that starts with a word $w \ne \texttt{</s>}$. Then
  $p(hwz) = p(h)\,p(w \mid h)\,p(z \mid hw) \le p(h)\,p(w \mid h) \le p(h)\,p(\texttt{</s>} \mid h)$.
- **(b)** The $|y|$ word steps contribute at most $q^{|y|}$, and the final $\texttt{</s>}$ at most 1. Then
  $q^{|y^\star|} \ge p(y^\star) \ge p(y')$; take logarithms, remembering that $\log q \lt 0$. $\square$

So the mode ends as soon as stopping is the single likeliest move, even if it has probability 0.1 against 0.9 spread
over many words. Theorem 4(a) is sufficient for an empty mode, not necessary. On the trigram, $\texttt{</s>}$ is the
argmax after only 0.037 of the prompts, yet 0.197 of the prompts have the empty mode.

**Normalization can have no optimum.** Take the model $p(w) = 0.9$, $p(\texttt{</s>}) = 0.1$ after every prefix, and
$\alpha = 1$. The string $w^n\texttt{</s>}$ scores $f(n) = 6(n \log 0.9 + \log 0.1)/(n + 6)$. Then
$f'(n) \propto 6\log 0.9 - \log 0.1 \gt 0$, because $0.1 \lt 0.9^6$. So $f$ rises toward $6 \log 0.9$ and never
reaches it: without a length cap, no string attains the supremum.

**Greedy as a regularized optimum.** Meister, Cotterell, and Vieira (2020) write beam search as the exact solution of
$\arg\max_y \log p(y) - \lambda R(y)$ as $\lambda \to \infty$. At $k = 1$ the regularizer is
$$R_{\text{greedy}}(y) = \sum_t \bigl(u_t(y_t) - \min_w u_t(w)\bigr)^2, \qquad u_t(w) = -\log p(w \mid y_{\lt t}).$$
This penalizes every step whose surprisal exceeds the locally best one, a form of uniform information density. On a
finite tie-free model the claim is short:
- $R_{\text{greedy}} \ge 0$, with equality only on the greedy string $g$.
- So for $\lambda \gt \max_{y \ne g} (\log p(y) - \log p(g))/R(y)$, which is a finite maximum, $g$ is the unique
  maximizer.

Their Theorem 3.2 extends this to beam($k$) with a set-valued regularizer; we cite it.

In [ ]:
bs.test_stopping_dominates()
bs.test_step_ceiling_bounds_the_mode_length()
bs.test_normalization_has_no_optimum()
bs.test_greedy_is_the_regularized_optimum()
print("normalized score of w^n </s>:", [round(bs.normalized_chain(0.9, n), 4) for n in (0, 1, 10, 100, 1000)],
      "→", round(6 * np.log(0.9), 4))

## 5. Search error against model error

On both engines, beam search reaches the mode as $k$ grows, and its strings shrink toward the mode's length.
- **Trigram.** Search error goes from 0.803 under greedy to 0 at $k = 64$.
- **Decoder.** It still makes search errors on 0.207 of the prompts at $k = 64$: without recombination, a width-64
  beam covers little of the tree.

The prompts' held-out sentences continue for 7.30 words on average. The trigram's mode stops after 2.37, and it is
empty for 0.197 of the prompts. A wider beam is a better search for a worse answer. Koehn and Knowles (2017) report
the same pattern in translation, and Stahlberg and Byrne (2019) found the empty translation to be the exact mode for
51.8% of their test sentences.

In [ ]:
bs.test_search_error_against_model_error()
bs.test_decoder_scores_match_forward()
for name in ("trigram", "decoder"):
    print(name)
    for r in bs.curve(name, 0.0)["rows"]:
        print(f"  k = {r['k']:2d}  search error {r['search_error']:.3f}  exact {r['exact']:.3f}  "
              f"length {r['length']:.3f}  copied {r['copied']:.3f}")
    print(f"  mode: length {bs.mode_stats(name)['length']:.3f}")

## 6. Non-termination

**Theorem 5 (greedy is eventually periodic).** On an order-$n$ Markov model, greedy decoding either emits
$\texttt{</s>}$ within $|S|$ steps or never does. Here $S$ is the set of states, with $|S| \le (|V| + 1)^{n-1}$.

*Proof.*
- Greedy's next token is a function of the state, so the states follow a deterministic map
  $\sigma_{t+1} = F(\sigma_t)$ on the finite set $S$.
- Among $\sigma_0, \dots, \sigma_{|S|}$ two must coincide, say $\sigma_i = \sigma_j$ with $i \lt j$. From then on
  the states, and so the tokens, repeat with period $j - i$.
- If $\texttt{</s>}$ has not appeared by step $j$, it never will. $\square$

On the trigram, greedy cycles on 0.282 of the prompts.

**Proposition (Welleck et al. 2020, a stationary instance).** Take $k$ words, each with probability $(1-\varepsilon)/k$,
and $\texttt{</s>}$ with probability $\varepsilon \lt (1 - \varepsilon)/k$, after every prefix. Then the model is
consistent, but beam($k$) never terminates.

*Proof.*
- **Consistency.** $\Pr(|Y| \ge t) = (1 - \varepsilon)^t \to 0$.
- **Non-termination.** At every step, the best open prefix's $k$ word-children score above every
  $\texttt{</s>}$-child, so $\texttt{</s>}$ is never kept. $\square$

Their Theorem 3.4 builds a consistent RNN with the same effect for every *incomplete* decoder: one that considers fewer
than $|V|$ tokens per step.

In [ ]:
bs.test_greedy_is_eventually_periodic()
bs.test_consistent_model_beam_never_ends()
for e in bs.loop_examples():
    print(e["prompt"], "|", " ".join(e["pre"]), "[", " ".join(e["cycle"]), "] repeats with period", e["period"])

## 7. Finance: the mode is boilerplate

The held-out MD&A text the prompts open costs 2.576 nats per word under the trigram. The trigram's exact mode costs
1.367: decoding returns far more predictable text than the filings themselves.

The mode is often empty. Length normalization makes outputs longer, but not more like the filings:
- At $\alpha = 1$, beam(16) pulls 0.133 of the prompts onto the tail of one 8-K template sentence: "... is filing this
  current report to disclose a material event."
- That sentence is the corpus's cheapest long path, and normalization rewards length.

Without a cap, this is the no-optimum example of §4.

In [ ]:
bs.test_the_mode_is_boilerplate()
print("reference NLL", round(bs.reference_nll(), 3), "| mode NLL", round(bs.mode_stats("trigram")["nll"], 3))
print("attractor:", bs.attractor())
for e in bs.examples("trigram"):
    print(e)

## The lab's data and the harness

In [ ]:
bs.test_viz_payload()
bs.test_viz_mirror_matches_payload()
bs.test_corpus_fingerprint()
print("payload sections:", list(bs.viz_payload()))

In [ ]:
assert bs._run_harness() == 0